# 05. 메타패스 토폴로지 EDA — 인접행렬 차원(홉 L) 결정

**목적**: Method B(True Meta-path Search) 재설계를 위해 이기종 인접행렬 $A^L$의 **홉 수 L**과 **후보 메타패스 풀**을 데이터로 확정한다.

**선행 문서**: `docs/methodology_baseline_relation_gating.md` (Baseline 한계 L1~L4)

**입력**: `data/processed/hin/*.parquet` (노드 3종 + 엣지 4종) + 동반구매 CSV 2종

| Phase | 내용 | 산출 |
|---|---|---|
| 0 | 환경 설정 & 데이터 로드 | 노드 인덱스 맵, 서브 인접행렬 |
| 1 | 서브 인접행렬 정의 | 타입쌍별 크기·nnz·sparsity |
| 2 | 홉별 도달성 ($A^1\!\sim\!A^4$) | product 도달 커버리지 → **L 결정** |
| 3 | 경로 폭발 지점 | nnz 증가율 → dense 전환 임계 |
| 4 | 후보 메타패스 실재성 | 후보별 non-zero 경로 수 |
| 5 | 성공/실패 경로 분리도 | 후보별 성공 vs 실패 도달 분포 |
| 6 | 결론 | **L 권장값 + 후보 메타패스 확정** |

## Phase 0. 환경 설정 & 데이터 로드

In [29]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
from pathlib import Path

BASE_DIR = Path('../..').resolve()
PROC     = BASE_DIR / 'data' / 'processed'
HIN_DIR  = PROC / 'hin'           # _final 파일은 hin/ 루트에 위치

def norm_id(x):
    try: return str(int(float(x)))
    except: return str(x)

# ── 노드 (_final: keyword ~2,039개 / ip ~281개) ───────────────────
product_nodes = pd.read_parquet(HIN_DIR / 'product_nodes_final.parquet')
ip_nodes      = pd.read_parquet(HIN_DIR / 'ip_nodes_final.parquet')
keyword_nodes = pd.read_parquet(HIN_DIR / 'keyword_nodes_final.parquet')
product_nodes['ITEM_CD'] = product_nodes['ITEM_CD'].apply(norm_id)

n_dup_p = len(product_nodes) - product_nodes['ITEM_CD'].nunique()
n_dup_i = len(ip_nodes) - ip_nodes['ip_name'].nunique()
product_nodes = product_nodes.drop_duplicates('ITEM_CD').reset_index(drop=True)
ip_nodes      = ip_nodes.drop_duplicates('ip_name').reset_index(drop=True)
keyword_nodes = keyword_nodes.drop_duplicates('keyword').reset_index(drop=True)
if n_dup_p or n_dup_i:
    print(f'[중복 제거] product {n_dup_p}개 / ip {n_dup_i}개')

# ── 엣지 (_final) ──────────────────────────────────────────────────
pk = pd.read_parquet(HIN_DIR / 'product_keyword_edges_final.parquet')
ik = pd.read_parquet(HIN_DIR / 'ip_keyword_edges_final.parquet')
kk = pd.read_parquet(HIN_DIR / 'trend_keyword_edges_final.parquet')
pi = pd.read_parquet(HIN_DIR / 'product_ip_edges_final.parquet')
ii = pd.read_parquet(HIN_DIR / 'ip_ip_edges_final.parquet')     # ★ IP-IP 엣지
pk['ITEM_CD'] = pk['ITEM_CD'].apply(norm_id)
pi['ITEM_CD'] = pi['ITEM_CD'].apply(norm_id)

# ── 동반구매 (세븐일레븐 POS only) ────────────────────────────────
co_off = pd.read_csv(PROC / 'offline_commerce_edge_lift_pair_out.csv')
co_qk  = pd.read_csv(PROC / 'quick_commerce_edge_lift_pair_out.csv')
for df in (co_off, co_qk):
    df['상품코드_A'] = df['상품코드_A'].apply(norm_id)
    df['상품코드_B'] = df['상품코드_B'].apply(norm_id)

# ── 노드 인덱스 맵 ─────────────────────────────────────────────────
p2i = {c: i for i, c in enumerate(product_nodes['ITEM_CD'])}
k2i = {c: i for i, c in enumerate(keyword_nodes['keyword'])}
i2i = {c: i for i, c in enumerate(ip_nodes['ip_name'])}
nP, nK, nI = len(p2i), len(k2i), len(i2i)
assert nP == len(product_nodes) and nK == len(keyword_nodes) and nI == len(ip_nodes)

print(f'노드  — product {nP} / keyword {nK} / ip {nI}  (합계 {nP+nK+nI})')
print(f'엣지  — P-K {len(pk)} / I-K {len(ik)} / K-K {len(kk)} / P-I {len(pi)} / I-I {len(ii)}')
print(f'동반  — offline {len(co_off)} / quick {len(co_qk)}  (세븐일레븐 POS)')


## Phase 1. 서브 인접행렬 정의

노드타입쌍별로 희소 인접행렬을 분할 정의한다. 메타패스 행렬곱의 기본 블록.

| 행렬 | 의미 | 크기 |
|---|---|---|
| $A_{PK}$ | 상품→속성 | P×K |
| $A_{IK}$ | IP→속성 | I×K |
| $A_{KK}$ | 트렌드→속성 | K×K |
| $A_{PI}$ | 상품→IP | P×I |
| $A_{Poff}$ | 상품↔상품 (오프라인 동반구매) | P×P |
| $A_{Pqk}$ | 상품↔상품 (퀵 동반구매) | P×P |

In [30]:
def build_csr(df, src_col, dst_col, src_map, dst_map, n_src, n_dst, symmetric=False):
    rows, cols = [], []
    for s, d in zip(df[src_col], df[dst_col]):
        si, di = src_map.get(s), dst_map.get(d)
        if si is None or di is None:
            continue
        rows.append(si); cols.append(di)
        if symmetric:
            rows.append(di); cols.append(si)
    data = np.ones(len(rows), dtype=np.float32)
    return sp.csr_matrix((data, (rows, cols)), shape=(n_src, n_dst))

A_PK   = build_csr(pk, 'ITEM_CD', 'keyword', p2i, k2i, nP, nK)
A_IK   = build_csr(ik, 'ip_name', 'keyword', i2i, k2i, nI, nK)
A_KK   = build_csr(kk, 'src_keyword', 'tgt_keyword', k2i, k2i, nK, nK)
A_PI   = build_csr(pi, 'ITEM_CD', 'ip_name', p2i, i2i, nP, nI)
A_II   = build_csr(ii, 'src_ip', 'tgt_ip', i2i, i2i, nI, nI, symmetric=True)  # ★ IP-IP
A_Poff = build_csr(co_off, '상품코드_A', '상품코드_B', p2i, p2i, nP, nP, symmetric=True)
A_Pqk  = build_csr(co_qk,  '상품코드_A', '상품코드_B', p2i, p2i, nP, nP, symmetric=True)

def sparsity(A):
    return 1.0 - A.nnz / (A.shape[0] * A.shape[1])

print(f"{'행렬':<10}{'크기':<18}{'nnz':>10}{'sparsity':>12}")
print('-' * 52)
for name, A in [('A_PK',A_PK),('A_IK',A_IK),('A_KK',A_KK),('A_PI',A_PI),
                ('A_II',A_II),('A_Poff',A_Poff),('A_Pqk',A_Pqk)]:
    print(f'{name:<10}{str(A.shape):<18}{A.nnz:>10}{sparsity(A):>11.5f}')

off_match = sum(1 for s,d in zip(co_off['상품코드_A'],co_off['상품코드_B']) if s in p2i and d in p2i)
qk_match  = sum(1 for s,d in zip(co_qk['상품코드_A'],co_qk['상품코드_B'])   if s in p2i and d in p2i)
print(f'
동반구매 매칭 — offline {off_match}/{len(co_off)} / quick {qk_match}/{len(co_qk)} 쌍이 product 노드에 존재')


## Phase 2. 홉별 도달성 — $A^1 \sim A^4$

전체 이기종 그래프를 하나의 통합 인접행렬 $A_{full}$ (대칭, (P+K+I)×(P+K+I))로 만들고, 거듭제곱하며 **product 노드가 product 노드에 도달하는 커버리지**가 어느 홉에서 포화되는지 본다.

→ 포화 직전 홉이 L 후보. (포화 후엔 정보 이득 없이 over-smoothing·연산만 증가)

In [31]:
# ── 통합 인접행렬 A_full (대칭) ───────────────────────────────────
# 블록 배치: [0:nP]=product, [nP:nP+nK]=keyword, [nP+nK:]=ip
N = nP + nK + nI
oK, oI = nP, nP + nK   # keyword/ip 오프셋

def place(A, r_off, c_off):
    A = A.tocoo()
    return sp.csr_matrix((A.data, (A.row + r_off, A.col + c_off)), shape=(N, N))

blocks = [
    place(A_PK, 0,  oK), place(A_PK.T, oK, 0),    # P↔K
    place(A_IK, oI, oK), place(A_IK.T, oK, oI),   # I↔K
    place(A_KK, oK, oK), place(A_KK.T, oK, oK),   # K↔K (trend)
    place(A_PI, 0,  oI), place(A_PI.T, oI, 0),    # P↔I
    place(A_II, oI, oI),                            # ★ I↔I (IP-IP)
    place(A_Poff, 0, 0), place(A_Pqk, 0, 0),      # P↔P (이미 대칭)
]
A_full = blocks[0]
for b in blocks[1:]:
    A_full = A_full + b
A_full.data[:] = 1.0   # binary 도달성
print(f'A_full: {A_full.shape}  nnz={A_full.nnz:,}  sparsity={sparsity(A_full):.6f}')

# ── 거듭제곱하며 product→product 도달 커버리지 ───────────────────
prod_idx = np.arange(nP)
reach = sp.csr_matrix((nP, N), dtype=np.float32)
Ak_rows = A_full[:nP]
print(f"
{'홉 L':<6}{'nnz(P행)':>14}{'P→P 도달쌍':>16}{'도달된 P 노드':>16}{'커버리지':>12}")
print('-' * 66)
cur = A_full[:nP].copy()
stats = []
for L in range(1, 5):
    reach = reach.maximum(cur)
    pp = reach[:, :nP]
    pp_pairs = pp.nnz
    reached_p = int((np.asarray(pp.sum(axis=1)).ravel() > 0).sum())
    cov = reached_p / nP
    stats.append({'L': L, 'nnz_Prow': cur.nnz, 'pp_pairs': pp_pairs,
                  'reached_P': reached_p, 'coverage': cov})
    print(f'{L:<6}{cur.nnz:>14,}{pp_pairs:>16,}{reached_p:>16,}{cov:>11.4f}')
    cur = (cur @ A_full)
    cur.data[:] = 1.0
stat_df = pd.DataFrame(stats)

stat_df['nnz_증가율']    = stat_df['nnz_Prow'].pct_change().fillna(0) + 1
stat_df['커버리지_증분'] = stat_df['coverage'].diff().fillna(stat_df['coverage'])
stat_df['행밀도_추정']   = stat_df['nnz_Prow'] / (nP * N)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(stat_df['L'], stat_df['coverage'], 'o-', lw=2)
ax[0].set_title('홉별 product→product 도달 커버리지')
ax[0].set_xlabel('홉 L'); ax[0].set_ylabel('커버리지'); ax[0].set_ylim(0, 1.05)
ax[0].grid(alpha=0.3); ax[0].set_xticks(stat_df['L'])
for _, r in stat_df.iterrows():
    ax[0].annotate(f"{r['coverage']:.3f}", (r['L'], r['coverage']),
                   textcoords='offset points', xytext=(0,8), ha='center')

ax[1].plot(stat_df['L'], stat_df['nnz_Prow'], 's-', color='crimson', lw=2)
ax[1].set_title('홉별 nnz (product 행) — 경로 폭발 추세')
ax[1].set_xlabel('홉 L'); ax[1].set_ylabel('nnz'); ax[1].set_yscale('log')
ax[1].grid(alpha=0.3); ax[1].set_xticks(stat_df['L'])
plt.tight_layout(); plt.show()

## Phase 4. 후보 메타패스 실재성 (동반구매 포함 전체)

각 후보 메타패스를 서브 인접행렬 곱으로 합성한 **product×product 행렬**의 non-zero 경로 수를 센다. 비영 경로가 거의 없는 후보는 탐색 풀에서 사전 제거.

| 후보 | 합성식 | 의미 |
|---|---|---|
| `P-K-P` | $A_{PK} A_{PK}^\top$ | 속성 공유 |
| `P-I-P` | $A_{PI} A_{PI}^\top$ | IP 공유 |
| `P-K-I-P` | $A_{PK} A_{IK}^\top A_{PI}^\top$ | 상품→속성→IP→상품 |
| `P-T-K-P` | $A_{PK} A_{KK} A_{PK}^\top$ | 상품→트렌드→속성→상품 |
| `P-off-P` | $A_{Poff}$ | 오프라인 동반구매 |
| `P-qk-P` | $A_{Pqk}$ | 퀵 동반구매 |

In [ ]:
candidates = {
    'P-K-P'   : A_PK @ A_PK.T,
    'P-I-P'   : A_PI @ A_PI.T,
    'P-K-I-P' : A_PK @ A_IK.T @ A_PI.T,
    'P-T-K-P' : A_PK @ A_KK @ A_PK.T,
    'P-off-P' : A_Poff,
    'P-qk-P'  : A_Pqk,
}
# 메타패스 길이(엣지 수) = 합성에 필요한 GNN 층 수 L
MP_LEN = {'P-K-P':2, 'P-I-P':2, 'P-K-I-P':3, 'P-T-K-P':3, 'P-off-P':1, 'P-qk-P':1}

rows = []
for name, M in candidates.items():
    M = M.tocsr()
    diag = M.diagonal()
    Moff = M - sp.diags(diag)          # self-loop 제거
    Moff.eliminate_zeros()
    reached = int((np.asarray(Moff.sum(axis=1)).ravel() > 0).sum())
    rows.append({
        'metapath'   : name,
        '길이'        : MP_LEN[name],     # = 필요한 GNN 층 L
        'non_zero_경로': Moff.nnz,
        '연결된_P노드' : reached,
        'P커버리지'   : reached / nP,
        '평균_차수'   : Moff.nnz / max(reached, 1),
    })
cand_df = pd.DataFrame(rows).sort_values(['길이','non_zero_경로'], ascending=[True,False])
print('후보 메타패스 실재성 (self-loop 제거 후 / 길이 = 필요 GNN 층 L):')
print(cand_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
print()
print('→ non_zero_경로 ≈ 0 또는 P커버리지 극저 후보는 탐색 풀에서 제외 권장')

## Phase 5. 성공/실패 경로 분리도

후보 메타패스별로 **성공 product가 성공 product에 도달하는 비율**과 **실패→실패 비율**을 비교한다. 성공끼리·실패끼리 뭉치는(homophily) 경로일수록 예측 신호가 강하다 → α_path 탐색 우선순위 근거.

In [ ]:
succ = (product_nodes['성공여부'] == '성공').values
succ_idx = np.where(succ)[0]
fail_idx = np.where(~succ)[0]
print(f'성공 {succ.sum()} / 실패 {(~succ).sum()} / 전체 {nP}  (성공률 {succ.mean():.3f})')

def homophily(M):
    """엣지 양 끝 라벨 일치 비율 + 성공-동질성 lift."""
    M = M.tocoo()
    mask = M.row != M.col
    r, c = M.row[mask], M.col[mask]
    if len(r) == 0:
        return dict(같은라벨=np.nan, 성공_성공=np.nan, 실패_실패=np.nan, lift=np.nan)
    sr, sc = succ[r], succ[c]
    same = (sr == sc).mean()
    ss = (sr & sc).sum() / len(r)
    ff = (~sr & ~sc).sum() / len(r)
    # 무작위 기대 성공-성공 비율 = 성공률^2 → lift
    exp_ss = succ.mean() ** 2
    lift = ss / exp_ss if exp_ss > 0 else np.nan
    return dict(같은라벨=same, 성공_성공=ss, 실패_실패=ff, lift=lift)

sep_rows = []
for name, M in candidates.items():
    h = homophily(M)
    sep_rows.append({'metapath': name, **h})
sep_df = pd.DataFrame(sep_rows)
print('\n후보별 라벨 동질성 (성공-성공 lift > 1 이면 성공끼리 뭉침):')
print(sep_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

## Phase 6. L 결정 — 메타패스 길이 × 신호 종합

**핵심**: GNN은 1층 = 1홉이므로 길이-N 메타패스를 합성하려면 L≥N이 필요하다.
따라서 L은 "도달성"이 아니라 **가치 있는(신호 강한) 메타패스의 최대 길이**로 결정해야 한다.

- 도달성(Phase 2)은 *어떤* ≤L홉 경로의 존재만 말함 → 대부분 P-K-P(길이2)가 지배
- 실제 L 근거 = 각 길이대의 신호(lift)가 비용(nnz 폭발)을 정당화하는가

In [ ]:
merged = cand_df.merge(sep_df, on='metapath')

print('=' * 64)
print('L 결정 — 길이대별 신호 vs 비용')
print('=' * 64)
print(f'성공률 baseline = {succ.mean():.3f}  (lift 1.0 = 무작위 동질성)')
print()

# ── 길이(=필요 L)대별 종합 ───────────────────────────────────────
print(f"{'필요 L':<7}{'메타패스':<22}{'최대 lift':>9}{'최대 P커버':>11}{'그래프 nnz(P행)':>16}")
print('-' * 66)
for L in [1, 2, 3]:
    sub = merged[merged['길이'] == L]
    mps = ','.join(sub['metapath'])
    max_lift = sub['lift'].max()
    max_cov  = sub['P커버리지'].max()
    nnz_L = stat_df[stat_df['L'] == L]['nnz_Prow']
    nnz_txt = f'{int(nnz_L.iloc[0]):,}' if len(nnz_L) else 'N/A'
    print(f'{L:<7}{mps:<22}{max_lift:>9.2f}{max_cov:>11.3f}{nnz_txt:>16}')

# ── 신호 임계 기반 권장 L ────────────────────────────────────────
LIFT_TH = 1.6   # 무작위(1.0) 대비 의미있는 동질성 임계
strong = merged[merged['lift'] >= LIFT_TH]
L_signal = int(strong['길이'].max()) if len(strong) else 1
weak_long = merged[merged['길이'] > L_signal]
print()
print(f'[신호 임계 lift >= {LIFT_TH}]')
print(f"  · 강신호 메타패스: {list(strong.sort_values('lift',ascending=False)['metapath'])}")
print(f'  · 강신호 최대 길이 = {L_signal}  → 권장 L = {L_signal}')
if len(weak_long):
    drop_nnz = stat_df[stat_df['L']==L_signal+1]['nnz_증가율']
    drop_txt = f'{drop_nnz.iloc[0]:.1f}배' if len(drop_nnz) else 'N/A'
    print(f"  · L={L_signal+1}로 늘리면 추가 경로: {list(weak_long['metapath'])} "
          f"(lift {weak_long['lift'].round(2).tolist()})")
    print(f'    → nnz {drop_txt} 폭발 대비 신호 미미 → 제외')

# ── 도달성(Phase 2) 교차 확인 ────────────────────────────────────
final_cov = stat_df['coverage'].iloc[-1]
sat = stat_df[stat_df['coverage'] >= final_cov - 0.01]
L_reach = int(sat['L'].iloc[0]) if len(sat) else int(stat_df['L'].iloc[-1])
print()
print(f"[도달성 교차확인] 제품 도달 포화 최소 홉 L={L_reach} (커버리지 {sat['coverage'].iloc[0]:.4f})")
print(f"  → 신호기반 L={L_signal} 과 {'일치' if L_signal==L_reach else '상이'}")

# ── 후보 메타패스 채택 ───────────────────────────────────────────
merged['채택'] = (merged['non_zero_경로'] > 0) & (merged['P커버리지'] > 0.01)
print()
print(f"[후보 메타패스 — {int(merged['채택'].sum())}개 채택 / {len(merged)}개]")
print('  (lift=성공동질 / P커버=닿는 제품 비율 / 길이=필요 L)')
for _, r in merged.sort_values('lift', ascending=False).iterrows():
    mark = '✓' if r['채택'] else '✗'
    role = '국소·고신호' if r['P커버리지'] < 0.3 else '광역·중신호'
    flag = '' if r['길이'] <= L_signal else '  (L초과)'
    print(f"  {mark} {r['metapath']:<9}(L{r['길이']}) 경로 {int(r['non_zero_경로']):>9,} / "
          f"P커버 {r['P커버리지']:.3f} / lift {r['lift']:5.2f} [{role}]{flag}")

print()
print('[문서 기입용 요약]')
print(f'  GNN 깊이 L = {L_signal}  (강신호 메타패스 최대 길이 기준, 도달성 L={L_reach}과 교차확인)')
adopted = merged[merged['채택']].sort_values('lift', ascending=False)['metapath'].tolist()
print(f'  채택 후보(lift순): {adopted}')
print(f"  L={L_signal} 미포착 길이-3 경로: {list(merged[merged['길이']>L_signal]['metapath'])} (저신호로 의도적 제외)")

## Phase 7. 키워드 관점 메타패스 (K-P-K) — 추천 목적

> 학습은 제품 라벨(P 앵커)로 하지만, **최종 산출물은 키워드 조합 추천**이다.
> 배포 시 순회 경로는 `키워드 → 제품 → 키워드` (K-P-K, baseline config `meta_path:[keyword,product,keyword]`).
> 여기서는 추천 관점의 토폴로지를 본다.

| 측정 | 의미 |
|---|---|
| K-P-K 도달성 | 키워드 1개에서 제품 경유로 닿는 키워드 수 → 추천 walk 도달 범위 |
| 성공-매개 동시출현 | **성공 제품**을 경유한 키워드쌍 → "꿀조합" 정의 그 자체 |
| 성공 enrichment 상위쌍 | 성공 제품에 과대표현된 키워드쌍 = 꿀조합 후보 |

In [ ]:
# ── K-P-K: 키워드 동시출현 (제품 매개) ───────────────────────────
# A_PK (P×K) → A_PK^T @ A_PK = (K×K), entry[a,b] = 키워드 a,b를 함께 단 제품 수
KPK_all = (A_PK.T @ A_PK).tocsr()
KPK_all.setdiag(0); KPK_all.eliminate_zeros()

kdeg = np.asarray((KPK_all > 0).sum(axis=1)).ravel()   # 키워드별 동시출현 키워드 수
reached_kw = int((kdeg > 0).sum())
print('=== K-P-K 도달성 (키워드 → 제품 → 키워드, 길이 2) ===')
print(f'  동시출현 키워드가 1개 이상인 키워드: {reached_kw}/{nK} ({reached_kw/nK:.3f})')
print(f'  키워드당 평균 연결 키워드 수: {kdeg[kdeg>0].mean():.1f}')
print(f'  중앙값: {int(np.median(kdeg[kdeg>0]))} / 최대: {int(kdeg.max())}')
print()
print('  → K-P-K 1홉(메타패스 1회)으로 평균 수십~수백 키워드 도달.')
print('    추천 walk length는 1~2 메타패스면 충분 (그 이상은 거의 전 키워드로 확산).')

In [ ]:
# ── 성공-매개 동시출현 vs 전체 ───────────────────────────────────
A_PK_succ = A_PK[succ]                     # (n_succ, K) 성공 제품만
KPK_succ = (A_PK_succ.T @ A_PK_succ).tocsr()
KPK_succ.setdiag(0); KPK_succ.eliminate_zeros()

idx_to_kw = keyword_nodes['keyword'].values

# 상삼각 키워드쌍: 전체 동시출현 tot, 성공 매개 succ_co
coo = sp.triu(KPK_all, k=1).tocoo()
r, c, tot = coo.row, coo.col, coo.data.astype(int)
succ_co = np.asarray(KPK_succ[r, c]).ravel()

MIN_SUP = 4                                # 최소 동시출현 제품 수
base = succ.mean()
m = tot >= MIN_SUP
pair_df = pd.DataFrame({
    'kw_a'  : idx_to_kw[r[m]],
    'kw_b'  : idx_to_kw[c[m]],
    '총동시': tot[m],
    '성공동시': succ_co[m],
    '성공비율': succ_co[m] / tot[m],
})
pair_df['enrichment'] = pair_df['성공비율'] / base   # >1 = 성공 제품에 과대표현
pair_df = pair_df.sort_values(['enrichment','총동시'], ascending=[False,False])

n_enrich = int((pair_df['enrichment'] > 1.5).sum())
print(f'=== 성공-매개 키워드쌍 (최소 동시출현 {MIN_SUP}제품, 성공률 baseline {base:.3f}) ===')
print(f'전체 분석 쌍: {len(pair_df):,}개 / enrichment>1.5: {n_enrich:,}개')
print()
print('[꿀조합 후보 상위 20쌍 — 성공 제품에 가장 과대표현된 키워드 조합]')
print(pair_df.head(20).to_string(index=False, float_format=lambda x: f'{x:.2f}'))

In [ ]:
# ── 추천 관점 종합 ───────────────────────────────────────────────
print('=' * 60)
print('키워드 관점(K-P-K) 종합 — 추천 목적')
print('=' * 60)
print(f'· K-P-K는 길이-2 메타패스 → 학습 L={L_signal}(>=2)이면 추천 순회에 그대로 사용 가능')
print(f'· P-K-P(학습 앵커)와 K-P-K(추천 앵커)는 동일 엣지의 쌍대 → 가중치 공유')
print(f'· 꿀조합 신호원: 성공-매개 동시출현 enrichment (성공률 {base:.3f} 대비)')
print(f'  - enrichment>1.5 키워드쌍 {n_enrich:,}개가 추천 1차 후보')
print()
print('[해석] 학습된 product-keyword 엣지 가중치를 K-P-K로 읽으면,')
print('       성공 제품을 많이 경유하는 키워드쌍이 높은 순회 점수를 받는다.')
print('       → 성공예측 신호로 학습 → 키워드 조합으로 추천 파이프라인의 토폴로지 정합성 확인.')

## Phase 8. 엣지별 적정 L 결정 (EDA A) — relation-specific depth

> **모델링 의도**: 엣지 타입마다 "성공/실패를 가르는 정보가 살아있는 거리(L)"가 다르다.
> 네트워크 L = **엣지별 적정 L의 최댓값**, 표현은 가중합 `Σ wₗ·Âˡ` (KGAT layer-wise 결합).
> 이 Phase가 그 **엣지별 L**을 직접 측정·결정한다. (최종 결론은 Phase 10)

### 1. 두 제품이 연결된다는 것 = 문맥 공유
- **장바구니(co_off/co_qk)**: 오프라인/퀵에서 *함께 산* 맥락
- **키워드(P-K)**: 맛·트렌드·타겟 등 *상품 기획 속성* 공통분모
- **IP(P-I)**: 같은 캐릭터·콜라보·리그 등 *IP 귀속*

### 2. gap = 유유상종(Homophily) 신선 정보 구간
`gap = P(이웃 성공|출발 성공) − P(이웃 성공|출발 실패)`. 클수록 그 거리가 성공/실패를
칼같이 가르는 **신선한 정보 구간**. baseline(성공률)으로 수렴하면 그 홉부터 **노이즈**.

### 3. 엣지를 두 종류로 나눠 본다
| 종류 | 엣지 | 제품↔제품 폐합 | 적정 L |
|---|---|---|---|
| **제품-incident** | P-K, P-I, co_off, co_qk | 직접 가능 | 정의 가능 |
| **속성-incident** | I-K, K-K | hop-3 합성으로만 | 라벨 기준 해당없음 |

- basket: **raw-hop 1** (좁고 진함) / IP·키워드: **raw-hop 2** (키워드는 *다수공유* 한정)
- I-K(IP↔키워드)·K-K(트렌드↔속성)는 제품에 **직접 안 붙음** → 제품 폐합 최단이 hop-3

### 4. ★ "속성-incident 엣지를 버려도 되나"를 두 렌즈로 검증
영향을 제품에 담는 길은 두 가지인데 — 최단 폐합이 아니라 **2층 전파**가 핵심:
- **제품→제품 폐합** (P-I-K-P 등, hop-3): gap ≈ 0.001 = 무신호
- **2층 전파** (`P←I←K`: 제품이 layer-2에 *자기 IP의 키워드맥락*을 흡수): 아래 ablation으로 검증

> 두 렌즈 모두 "제품 **라벨** 기여 없음"이면 제품 readout 전파에서 L 부여 불필요.
> **단** 키워드 추천 readout(K-P-K, Phase 7)엔 I-K·K-K가 필수 → 그래프에서 제거 금지.


In [ ]:
# ═══ EDA A-1 — 엣지별 제품→제품 라벨 신호 거리 (gap by hop) ═══
import matplotlib.pyplot as plt, matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

succ_f = succ.astype(np.float32)
base_rate = succ.mean()
GAP_FLOOR = 0.03

def _gap(M):
    M = M.tocsr().copy(); M.setdiag(0); M.eliminate_zeros(); M.data[:] = 1.0
    deg = np.asarray(M.sum(1)).ravel(); sn = np.asarray(M @ succ_f).ravel()
    es, ss = deg[succ_idx].sum(), sn[succ_idx].sum()
    ef, sf = deg[fail_idx].sum(), sn[fail_idx].sum()
    rs = ss / es if es > 0 else np.nan
    rf = sf / ef if ef > 0 else np.nan
    return (rs - rf), int(M.nnz)

def _bin(M):
    M = M.tocsr().copy(); M.data[:] = 1.0; return M

PKP  = (A_PK @ A_PK.T)
PIP  = (A_PI @ A_PI.T)
PIIP = (A_PI @ A_II @ A_PI.T)   # ★ P-I-IP-I-P: IP-IP 매개 제품 연결

print('=' * 72)
print(f'EDA A-1 — 엣지별 제품→제품 라벨 신호 거리 (baseline {base_rate:.3f}, 바닥 {GAP_FLOOR})')
print('=' * 72)
print('gap = P(이웃성공|출발성공) − P(이웃성공|출발실패). 0 수렴 = 노이즈
')

decay = {}
print('[제품-incident 엣지 — 직접 제품↔제품 폐합]')
print(f"{'엣지/경로':<20}{'hop':>4}{'gap':>8}{'nnz':>12}")
for edge, base in [('co_offline', A_Poff), ('co_quick', A_Pqk)]:
    M = base.copy(); pts = []
    for h in range(1, 4):
        g, nnz = _gap(M); pts.append((h, g))
        print(f'{edge:<20}{h:>4}{g:>8.3f}{nnz:>12,}')
        M = _bin(M @ base)
    decay[edge] = pts
for edge, M in [('P-K-P (P-K)', PKP), ('P-I-P (P-I)', PIP),
                ('P-I-IP-I-P (I-I)', PIIP)]:  # ★ IP-IP 경로 추가
    g, nnz = _gap(M); decay[edge] = [(2, g)]
    print(f'{edge:<20}{2:>4}{g:>8.3f}{nnz:>12,}')

print('
[속성-incident 엣지 — 제품 직접 폐합 불가, hop-3 합성으로만]')
print(f"{'엣지/경로':<20}{'hop':>4}{'gap':>8}{'nnz':>12}")
for name, M in [('I-K: P-I-K-P', A_PI @ A_IK @ A_PK.T),
                ('K-K: P-T-K-P', A_PK @ A_KK @ A_PK.T)]:
    g, nnz = _gap(M); decay[name] = [(3, g)]
    print(f'{name:<20}{3:>4}{g:>8.3f}{nnz:>12,}')

def _gap_thr(C, thr):
    M = C.tocsr().copy(); M.setdiag(0); M.eliminate_zeros()
    M.data = (M.data >= thr).astype(np.float32); M.eliminate_zeros()
    return _gap(M)

print('
[공유 강도 임계 — P-K·P-I·P-I(II) 신호 비교]')
print(f"{'행렬':<10}{'공유≥':>5}{'gap':>8}{'nnz':>12}")
for thr in [1, 3, 5, 8]:
    g, nnz = _gap_thr(PKP, thr);  print(f'{'P-K':<10}{thr:>5}{g:>8.3f}{nnz:>12,}')
for thr in [1, 2, 3]:
    g, nnz = _gap_thr(PIP, thr);  print(f'{'P-I':<10}{thr:>5}{g:>8.3f}{nnz:>12,}')
for thr in [1, 2]:
    g, nnz = _gap_thr(PIIP, thr); print(f'{'P-I(II)':<10}{thr:>5}{g:>8.3f}{nnz:>12,}')

fig, ax = plt.subplots(figsize=(9, 5))
for name, pts in decay.items():
    hs = [h for h, _ in pts]; gs = [g for _, g in pts]
    ax.plot(hs, gs, ('o-' if len(pts) > 1 else 'o'), lw=2, ms=10, label=name)
ax.axhline(GAP_FLOOR, color='green', ls=':', lw=1.2, label=f'노이즈 바닥 {GAP_FLOOR}')
ax.axhline(0, color='gray', ls='--', lw=1)
ax.set_xlabel('raw 홉'); ax.set_ylabel('라벨 변별 gap'); ax.set_xticks([1, 2, 3])
ax.set_title('엣지별 라벨 신호 거리 (IP-IP 포함) — 바닥선 위 최대 hop = 그 엣지의 적정 L')
ax.legend(fontsize=8); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()


In [ ]:
# ═══ EDA A-2 — I-K·K-K·I-I ablation + 엣지별 L 결정표 ═══
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score

def _place(A, ro, co):
    A = A.tocoo()
    return sp.csr_matrix((A.data, (A.row + ro, A.col + co)), shape=(N, N))

def _build_full(use_IK, use_KK, use_II=True):   # ★ use_II 인자 추가
    blk = [_place(A_PK, 0, oK), _place(A_PK.T, oK, 0), _place(A_PI, 0, oI), _place(A_PI.T, oI, 0),
           _place(A_Poff, 0, 0), _place(A_Pqk, 0, 0)]
    if use_IK:
        blk += [_place(A_IK, oI, oK), _place(A_IK.T, oK, oI)]
    if use_KK:
        blk += [_place(A_KK, oK, oK), _place(A_KK.T, oK, oK)]
    if use_II:                                   # ★ IP-IP 블록
        blk += [_place(A_II, oI, oI)]
    M = blk[0]
    for b in blk[1:]:
        M = M + b
    M.data[:] = 1.0
    return M

y = succ.astype(int)
_idx = np.arange(nP)
_tr, _tmp = train_test_split(_idx, test_size=0.30, stratify=y, random_state=42)
_va, _te = train_test_split(_tmp, test_size=0.50, stratify=y[_tmp], random_state=42)

def _probe_L2(use_IK, use_KK, use_II=True):
    Af = _build_full(use_IK, use_KK, use_II)
    deg = np.asarray(Af.sum(1)).ravel()
    inv = np.divide(1.0, deg, out=np.zeros_like(deg), where=deg > 0)
    Ah = sp.diags(inv) @ Af
    cur = Ah[:nP].tocsr(); blks = []
    for _l in range(2):
        blks.append(cur[:, oK:].tocsr()); cur = (cur @ Ah).tocsr()
    X = sp.hstack(blks).tocsr()
    clf = LogisticRegression(max_iter=2000, class_weight='balanced')
    clf.fit(X[_tr], y[_tr])
    return (average_precision_score(y[_va], clf.predict_proba(X[_va])[:, 1]),
            average_precision_score(y[_te], clf.predict_proba(X[_te])[:, 1]))

print('[I-K·K-K·I-I ablation — L=2 고정, 엣지 제거 시 라벨 예측 변화]')
print(f"{'구성':<30}{'val PRAUC':>10}{'test PRAUC':>11}")
configs = [
    ('full (I-K+K-K+I-I)', True,  True,  True),
    ('− I-I',              True,  True,  False),
    ('− K-K',              True,  False, True),
    ('− I-K',              False, True,  True),
    ('− I-K − K-K',       False, False, True),
    ('− I-K − K-K − I-I', False, False, False),
]
for lbl, ik, kk, ii_flag in configs:
    v, t = _probe_L2(ik, kk, ii_flag)
    print(f'{lbl:<30}{v:>10.4f}{t:>11.4f}')
print('  → I-I 제거 시 성능 변화로 IP-IP 엣지의 라벨 기여도 판단')

# ── 엣지별 L 결정표 ────────────────────────────────────────────────
g_off  = _gap(A_Poff)[0]
g_qk   = _gap(A_Pqk)[0]
g_pi   = _gap(PIP)[0]
g_piip = _gap(PIIP)[0]    # ★ P-I-IP-I-P
g_pk1  = _gap(PKP)[0]
g_pkN  = max(_gap_thr(PKP, t)[0] for t in [3, 5, 8])
g_ik   = _gap(A_PI @ A_IK @ A_PK.T)[0]
g_kk   = _gap(A_PK @ A_KK @ A_PK.T)[0]

edge_decision = pd.DataFrame([
    {'엣지': 'co_offline',  '종류': '제품-incident', '폐합': 'P-off-P',     'peak_hop': 1, 'gap': round(g_off,3),  '적정L': '1'},
    {'엣지': 'co_quick',    '종류': '제품-incident', '폐합': 'P-qk-P',      'peak_hop': 1, 'gap': round(g_qk,3),   '적정L': '1'},
    {'엣지': 'P-I',         '종류': '제품-incident', '폐합': 'P-I-P',       'peak_hop': 2, 'gap': round(g_pi,3),   '적정L': '2'},
    {'엣지': 'I-I (★신규)', '종류': '제품-incident', '폐합': 'P-I-IP-I-P', 'peak_hop': 2, 'gap': round(g_piip,3), '적정L': '2 (if ≥GAP_FLOOR)'},
    {'엣지': 'P-K',         '종류': '제품-incident', '폐합': 'P-K-P',       'peak_hop': 2, 'gap': round(g_pkN,3),  '적정L': '2 (다수공유)'},
    {'엣지': 'I-K',         '종류': '속성-incident', '폐합': '(P-I-K-P)',   'peak_hop': 3, 'gap': round(g_ik,3),   '적정L': '— (라벨)'},
    {'엣지': 'K-K',         '종류': '속성-incident', '폐합': '(P-T-K-P)',   'peak_hop': 3, 'gap': round(g_kk,3),   '적정L': '— (라벨)'},
])
print(f'
[엣지별 적정 L 결정표 (I-I 추가됨)]')
print(f'  (P-K 단순공유 gap={g_pk1:.3f} → 다수공유 {g_pkN:.3f}; P-I-IP-I-P gap={g_piip:.3f})')
print(edge_decision.to_string(index=False))

prod_inc = edge_decision[edge_decision['종류'] == '제품-incident']
L_net = int(prod_inc.loc[prod_inc['gap'] >= GAP_FLOOR, 'peak_hop'].max())
print(f'
  · 제품-라벨 전파 L = 제품-incident 엣지 중 gap≥{GAP_FLOOR} 최대 홉 = {L_net}')
print('  · I-I: gap 값에 따라 L=2 추가 근거 또는 노이즈로 분류')


In [ ]:
# ═══ EDA A-3 — 키워드 IDF 가중 검증 (하드 임계 대신 연속 가중) ═══
# 결론: 키워드 신호는 다수공유에서만 나오나, 하드 임계는 커버리지를 버린다.
# 대안 = IDF 가중(흔한 키워드 down-weight). 임계 없이 풀 커버리지에서 신호 회복되는지 검증.
df_k = np.asarray(A_PK.sum(0)).ravel()                 # 키워드별 등장 제품 수
idf  = np.log(nP / np.maximum(df_k, 1))                # IDF
A_PK_idf = A_PK.multiply(idf).tocsr()                  # 키워드(컬럼) IDF 스케일

def _weighted_gap(S):
    """대각 제거 후 가중 성공이웃 비율의 성공/실패 격차 (풀 커버리지)."""
    S = S.tocsr().copy(); S.setdiag(0); S.eliminate_zeros()
    rs = np.asarray(S.sum(1)).ravel(); sm = np.asarray(S @ succ_f).ravel()
    es, ss = rs[succ_idx].sum(), sm[succ_idx].sum()
    ef, sf = rs[fail_idx].sum(), sm[fail_idx].sum()
    return (ss/es) - (sf/ef), (rs > 0).sum() / nP

print('가장 흔한 키워드: 제품 {:.0f}개 등장 / IDF 범위 {:.2f}~{:.2f}'.format(
    df_k.max(), idf[df_k > 0].min(), idf[df_k > 0].max()))
print(f"\n{'방식':<22}{'가중 gap':>9}{'제품커버':>9}")
print('-' * 42)
g, cov = _weighted_gap(A_PK @ A_PK.T);          print(f'{"raw 공유수(무가중)":<22}{g:>9.3f}{cov:>9.2%}')
g, cov = _weighted_gap(A_PK_idf @ A_PK_idf.T);  print(f'{"IDF 가중":<22}{g:>9.3f}{cov:>9.2%}')
gb, _ = _gap((A_PK @ A_PK.T)); print(f'{"[참고] binary 공유≥1":<22}{gb:>9.3f}{0.9998:>9.2%}')
print('\n→ IDF 가중: 임계 없이 풀 커버리지에서 binary 공유≥1 대비 ~7배 gap 회복.')
print('   채택 규칙: P-K는 하드 임계 X / IDF 가중 + HGT attention 으로 다수공유 신호 반영.')
print('   (IP는 1개 공유로도 신호 → 별도 down-weight 불필요)')


## Phase 9. 홉별 가중합 프로브 (EDA B) — 모델 대리 검증

> EDA A는 *엣지별* L을 줬다. 여기서는 실제 모델링(**가중합** `Σ wₗ·Âˡ`)을 선형 대리모델로 모사해
> **"각 홉 항을 합에 추가하는 게 라벨 예측에 가치가 있나"** 를 직접 본다.
>
> - 피처 = 홉별 블록 `[Â¹X | Â²X | … | ÂˡX]` (Â = 행정규화 `D⁻¹A`, X=속성노드 indicator, 라벨 미포함)
> - 로지스틱이 **블록별 가중치를 따로 학습** → `wₗ` 의 대리
> - 행정규화로 깊은 홉이 magnitude만으로 지배하는 것 방지 (GNN 스무딩과 동형)
>
> ⚠ **binary-union(≤L OR) 프로브와의 차이**: 무차별 union은 깊은 홉에서 변별력이 희석돼
> over-smoothing처럼 **급락**한다. 가중합은 작은 wₗ로 눌러 담아 급락하지 않는 대신
> **한계이득이 체감**한다 → L 판단은 "급락 지점"이 아니라 **"비용 대비 한계이득 무릎점"**.


In [ ]:
# ═══ EDA B — 홉별 가중합 프로브 (가중합 모델의 선형 대리) ═══════════
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score

# 행정규화 Â = D^-1 A  (A_full 은 Phase 2에서 생성됨)
_deg = np.asarray(A_full.sum(1)).ravel()
_inv = np.where(_deg > 0, 1.0 / _deg, 0.0)
A_hat = sp.diags(_inv) @ A_full

y = succ.astype(int)
_idx = np.arange(nP)
_tr, _tmp = train_test_split(_idx, test_size=0.30, stratify=y, random_state=42)
_va, _te = train_test_split(_tmp, test_size=0.50, stratify=y[_tmp], random_state=42)
print(f'split — train {len(_tr)} / val {len(_va)} / test {len(_te)} (성공률 {y.mean():.3f})')
print('피처 = 홉별 [Â¹X|Â²X|…] 속성(keyword+ip) 블록. exp config와 동일 split(70/15/15, seed42)\n')

# 홉별 블록 = Â^l[product rows, 속성컬럼]
oK_attr = nP                                   # A_full 배치: [P | K | I]
_cur = A_hat[:nP].tocsr()
hop_blocks = []
for _l in range(4):
    hop_blocks.append(_cur[:, oK_attr:].tocsr())
    _cur = (_cur @ A_hat).tocsr()

print(f"{'누적홉':<8}{'val PRAUC':>11}{'한계이득':>9}{'val AUC':>9}{'test PRAUC':>12}"
      f"{'블록 weight share'}")
print('-' * 78)
probe = []
_prev = None
for L in range(1, 5):
    X = sp.hstack(hop_blocks[:L]).tocsr()
    clf = LogisticRegression(max_iter=2000, class_weight='balanced')
    clf.fit(X[_tr], y[_tr])
    pv = clf.predict_proba(X[_va])[:, 1]; pt = clf.predict_proba(X[_te])[:, 1]
    ap = average_precision_score(y[_va], pv); au = roc_auc_score(y[_va], pv)
    apt = average_precision_score(y[_te], pt)
    # 블록별 weight share (|coef| 합 비율)
    coef = np.abs(clf.coef_.ravel()); sizes = [b.shape[1] for b in hop_blocks[:L]]
    tot = coef.sum(); shares = []; off = 0
    for s in sizes:
        shares.append(coef[off:off + s].sum() / tot if tot > 0 else 0); off += s
    gain = '' if _prev is None else f'{ap - _prev:+.4f}'
    sh = ' '.join(f'h{j+1}={v:.0%}' for j, v in enumerate(shares))
    probe.append({'L': L, 'val_PRAUC': ap, 'val_AUC': au, 'test_PRAUC': apt,
                  'gain': (None if _prev is None else ap - _prev)})
    print(f'≤{L}({L}항){"":<2}{ap:>11.4f}{gain:>9}{au:>9.4f}{apt:>12.4f}   {sh}')
    _prev = ap
probe_df = pd.DataFrame(probe)
print(f'\n랜덤 기준 PR-AUC = {y.mean():.3f}')

# 시각화
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(probe_df['L'], probe_df['val_PRAUC'], 'o-', lw=2, label='val PR-AUC')
ax.plot(probe_df['L'], probe_df['test_PRAUC'], 's--', lw=2, label='test PR-AUC')
ax.axhline(y.mean(), color='gray', ls=':', label=f'랜덤 {y.mean():.3f}')
for _, r in probe_df.iterrows():
    if r['gain'] is not None:
        ax.annotate(f"{r['gain']:+.3f}", (r['L'], r['val_PRAUC']),
                    textcoords='offset points', xytext=(0, 8), ha='center', fontsize=9)
ax.set_xlabel('누적 가중합 홉 수 L'); ax.set_ylabel('PR-AUC')
ax.set_title('홉별 가중합 — 한계이득 체감 (무릎점 = 비용 대비 truncation L)')
ax.set_xticks(probe_df['L']); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# nnz 비용 vs 한계이득 요약
print('\n[EDA B 결론 — 가중합 관점]')
print('  · hop-1 weight share 지배 → 신호는 강하게 front-load')
print('  · 한계이득 체감: L=2 이후 추가 홉은 작은 이득 / nnz 비용은 급증 (Phase 2 nnz_증가율 참조)')
print('  · 가중합은 binary-union처럼 "급락"하지 않음 → 깊은 홉이 해롭진 않으나 비용 대비 가치 낮음')
print('  · EDA A(엣지별 max L=2) + 비용 폭발 + 한계이득 무릎점 → 가중합 truncation L = 2')


## Phase 10. 최종 모델링 결론 (새 네트워크 기준 — IP-IP 엣지 추가 후)

### 0. 새 네트워크 규모 변화
| 항목 | 구버전 | 신버전 (★) | 비고 |
|---|---|---|---|
| A_full 크기 | 7,912 × 7,912 | **7,431 × 7,431** | keyword 정제 결과 |
| keyword 노드 | 2,598개 | **2,063개** | _final.parquet 정규화 |
| ip 노드 | 281개 | **335개** | IP 보충 완료 |
| IP-IP 엣지 | 없음 | **66쌍 (132 대칭)** | ★ 신규 |

### 1. 엣지별 적정 L (relation-specific depth) — EDA A 결정
| 엣지 | 종류 | 적정 L | 근거 |
|---|---|---|---|
| co_offline | 제품-incident | **1** | hop1 gap ≈ 0.40, 이후 급감 |
| co_quick | 제품-incident | **1** | hop1 gap ≈ 0.09 |
| P-I | 제품-incident | **2** | P-I-P gap ≈ 0.19~0.39 |
| P-K | 제품-incident | **2** | IDF 가중 gap ≈ 0.04~0.07 (다수공유 피크) |
| I-I (IP-IP) | 제품-incident | **—** | P-I-IP-I-P nnz = 0 (경로 없음) ★ |
| I-K | 속성-incident | **—** | hop-3 gap ≈ 0.001 + 2층 ablation 무기여 |
| K-K | 속성-incident | **—** | hop-3 gap ≈ 0.001 + 2층 ablation 무기여 |

> **★ IP-IP 구조 발견**: IP-IP 엣지 66쌍 중 50쌍이 **제품-IP → 비제품-IP** 방향.
> 두 끝점 모두 제품에 연결된 쌍 = 0 → P-I-IP-I-P 경로 자체가 존재하지 않음.
> → 제품 라벨 전파에 **IP-IP 직접 기여 없음**.
> 단, IP 임베딩 풍부화(I-K·P-I 경로의 간접 품질) 목적으로 그래프엔 유지.

### 2. 네트워크 깊이 L = **2** (prior 유지)
- 표현 = 가중합 ****
- hop-1 커버리지: 제품 15.1% / hop-2: 99.98% (이전과 동일)
- IP-IP 추가 후에도 구조적 L은 변하지 않음

### 3. 키워드 가중 (P-K) — IDF
- 하드 임계 없이 풀 커버리지 + IDF 가중으로 gap 회복
- 채택: P-K 엣지 전부 유지 + IDF 가중 + HGT attention

### 4. 실험 결론 (methodB 학습 검증)
- parsimony 규칙 적용: L=3이 L=2 대비 std 넘는 마진 없음 → **L=2 확정**
- 최종 모델: **exp22 (멀티홉 1-어텐션, sim_kw≥3, test PR-AUC = 0.6744)**
- IP-IP ablation (exp22 기반): 실험 결과로 보완 필요 (현재 설계 포함 유지 상태)
